In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (5000, 9)

Column names:
['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']

Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif

In [7]:
print("Q1 - Industry mode:")
print(df["industry"].mode().tolist())

Q1 - Industry mode:
['technology']


In [8]:
numeric = df.select_dtypes(include="number")
corr = numeric.corr()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

print("\nQ2 - Correlations:")
for a, b in pairs:
    if a in corr.columns and b in corr.columns:
        print(a, "and", b, ":", corr.loc[a, b])


Q2 - Correlations:
interaction_count and lead_score : 0.9287123432397134
number_of_courses_viewed and lead_score : 0.769352062184664
number_of_courses_viewed and interaction_count : 0.7216090038937035
annual_income and interaction_count : 0.10637458469219016


In [9]:
target_candidates = [
    "converted", "conversion", "enrolled",
    "target", "lead_score"
]
target = next(
    (c for c in target_candidates if c in df.columns),
    None
)

print("\nTarget candidates:", target)
print("Columns:", df.columns.tolist())

if target is not None:
    features = [
        c for c in
        ["industry", "location", "lead_source", "employment_status"]
        if c in df.columns
    ]
    mi_df = df[features].copy()
    for c in features:
        mi_df[c] = mi_df[c].fillna("NA").astype(str)

    mi_encoded = pd.get_dummies(mi_df)
    mi = mutual_info_classif(
        mi_encoded, df[target], random_state=42
    )
    print("\nQ3 - Mutual information:")
    print(pd.Series(mi, index=mi_encoded.columns)
          .groupby(lambda x: x.split("_")[0])
          .sum().sort_values(ascending=False))

# Check target type and values before model training
print("\nTarget distribution:")
if target is not None:
    print(df[target].value_counts(dropna=False))



Target candidates: converted
Columns: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']

Q3 - Mutual information:
employment    0.034436
lead          0.028484
location      0.024526
industry      0.009444
dtype: float64

Target distribution:
converted
1    2883
0    2117
Name: count, dtype: int64


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Prepare data
X = df.drop(columns=["converted"])
y = df["converted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cat_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()
num_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor = ColumnTransformer([
    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore")
    ), cat_features),
    ("num", make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler()
    ), num_features)
])

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=2000)
)
model.fit(X_train, y_train)
print("Q4 Accuracy:", round(
    accuracy_score(y_test, model.predict(X_test)), 2
))


for feature in [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]:
    single_preprocessor = ColumnTransformer([
        ("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore")
        ), [feature])
    ]) if feature == "lead_source" else ColumnTransformer([
        ("num", make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ), [feature])
    ])

    single_model = make_pipeline(
        single_preprocessor,
        LogisticRegression(max_iter=2000)
    )
    single_model.fit(X_train[[feature]], y_train)
    print("Q5", feature, "Accuracy:", round(
        accuracy_score(
            y_test, single_model.predict(X_test[[feature]])
        ), 4
    ))


for c in [0.000001, 0.00001, 0.0001, 0.001]:
    tuned_model = make_pipeline(
        preprocessor,
        LogisticRegression(C=c, max_iter=2000)
    )
    tuned_model.fit(X_train, y_train)
    print("Q6 C =", c, "Accuracy:", round(
        accuracy_score(y_test, tuned_model.predict(X_test)), 4
    ))


/tmp/ipykernel_7176/3714891866.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(


Q4 Accuracy: 0.72
Q5 lead_source Accuracy: 0.628
Q5 number_of_courses_viewed Accuracy: 0.696
Q5 interaction_count Accuracy: 0.711
Q6 C = 1e-06 Accuracy: 0.6
Q6 C = 1e-05 Accuracy: 0.6
Q6 C = 0.0001 Accuracy: 0.657
Q6 C = 0.001 Accuracy: 0.726
